# NOTEBOOK : 07_select_best_models.ipynb
# ROLE     : Evaluate all 5 trained models on val.csv, rank by Log Loss, and save top 2 models.
# INPUT    : ../../data/processed/val.csv, ../artifacts/models/*.joblib, ../artifacts/features_list.json
# OUTPUT   : ../artifacts/best_two_models.json, ../artifacts/val_comparison.csv

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    root_mean_squared_error,
)

PROCESSED_DIR = Path("../../data/processed")
ARTIFACTS_DIR = Path("../artifacts")
MODELS_DIR = ARTIFACTS_DIR / "models"

val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
with open(ARTIFACTS_DIR / "features_list.json", "r", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

scaler = joblib.load(MODELS_DIR / "match_scaler.joblib")

X_val = val_df[FEATURE_COLS]
X_val_scaled = scaler.transform(X_val)
y_val = val_df["target"].values  # 0=A, 1=D, 2=H

# One-hot encode y_val for multi-class probability MSE / RMSE
y_val_onehot = np.eye(3)[y_val]

print(f"Validation set: {X_val.shape[0]} matches")

Validation set: 190 matches


In [2]:
MODEL_CONFIGS = [
    ("logistic_regression", True),
    ("decision_tree", False),
    ("random_forest", False),
    ("gradient_boosting", False),
    ("xgboost", False),
]

results = []

for name, use_scaled in MODEL_CONFIGS:
    model = joblib.load(MODELS_DIR / f"{name}.joblib")
    X_in = X_val_scaled if use_scaled else X_val

    y_prob = model.predict_proba(X_in)
    y_pred = model.predict(X_in)

    # Per-class recall (class 1 = Draw)
    class_recalls = recall_score(y_val, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    draw_recall = float(class_recalls[1])

    # Multi-class probability MSE & RMSE
    mse = float(mean_squared_error(y_val_onehot, y_prob))
    rmse = float(root_mean_squared_error(y_val_onehot, y_prob))

    results.append({
        "model": name,
        "uses_scaled": use_scaled,
        "log_loss": round(float(log_loss(y_val, y_prob, labels=[0, 1, 2])), 4),
        "accuracy": round(float(accuracy_score(y_val, y_pred)), 4),
        "precision": round(float(precision_score(y_val, y_pred, average="macro", zero_division=0)), 4),
        "recall": round(float(recall_score(y_val, y_pred, average="macro", zero_division=0)), 4),
        "f1_score": round(float(f1_score(y_val, y_pred, average="macro", zero_division=0)), 4),
        "mse": round(mse, 4),
        "rmse": round(rmse, 4),
        "draw_recall": round(draw_recall, 4),
    })

results_df = pd.DataFrame(results).sort_values("log_loss", ascending=True).reset_index(drop=True)
results_df.index = results_df.index + 1
results_df.index.name = "rank"

display(results_df)

,model,uses_scaled,log_loss,accuracy,precision,recall,f1_score,mse,rmse,draw_recall
rank,,,,,,,,,,
1,random_forest,False,1.0049,0.5158,0.3401,0.4400,0.3824,0.1995,0.4461,0.0000
2,logistic_regression,True,1.0192,0.5053,0.3329,0.4364,0.3776,0.2028,0.4496,0.0000
3,xgboost,False,1.0203,0.4789,0.3159,0.4071,0.3549,0.2024,0.4493,0.0000
4,gradient_boosting,False,1.0329,0.4842,0.4038,0.4144,0.3709,0.2050,0.4521,0.0217
5,decision_tree,False,1.5893,0.4421,0.3398,0.3858,0.3456,0.2138,0.4619,0.0217


In [3]:
# Save full validation comparison table
val_table_path = ARTIFACTS_DIR / "val_comparison.csv"
results_df.to_csv(val_table_path)
print(f"Saved validation comparison table to {val_table_path}")

# Select Top 2 by lowest Log Loss
top_two = results_df.head(2)
best_two_payload = {
    "selection_metric": "log_loss",
    "top_two_models": [
        {
            "rank": int(rank),
            "model": row["model"],
            "uses_scaled": bool(row["uses_scaled"]),
            "val_log_loss": float(row["log_loss"]),
            "val_accuracy": float(row["accuracy"]),
            "val_precision": float(row["precision"]),
            "val_recall": float(row["recall"]),
            "val_f1_score": float(row["f1_score"]),
            "val_mse": float(row["mse"]),
            "val_rmse": float(row["rmse"]),
        }
        for rank, row in top_two.iterrows()
    ],
}

best_two_path = ARTIFACTS_DIR / "best_two_models.json"
with open(best_two_path, "w", encoding="utf-8") as f:
    json.dump(best_two_payload, f, indent=2)

print(f"Saved top 2 models to {best_two_path}:")
print(json.dumps(best_two_payload, indent=2))

Saved validation comparison table to ..\artifacts\val_comparison.csv
Saved top 2 models to ..\artifacts\best_two_models.json:
{
  "selection_metric": "log_loss",
  "top_two_models": [
    {
      "rank": 1,
      "model": "random_forest",
      "uses_scaled": false,
      "val_log_loss": 1.0049,
      "val_accuracy": 0.5158,
      "val_precision": 0.3401,
      "val_recall": 0.44,
      "val_f1_score": 0.3824,
      "val_mse": 0.1995,
      "val_rmse": 0.4461
    },
    {
      "rank": 2,
      "model": "logistic_regression",
      "uses_scaled": true,
      "val_log_loss": 1.0192,
      "val_accuracy": 0.5053,
      "val_precision": 0.3329,
      "val_recall": 0.4364,
      "val_f1_score": 0.3776,
      "val_mse": 0.2028,
      "val_rmse": 0.4496
    }
  ]
}
